# Testing blop uing the 0.8.1 tutorial
khchan@lbl.gov, xchong@lbl.gov, awojdyla@lbl.gov; Nov 2025

#environment als_bl531

blop 0.8.1
Following https://nsls-ii.github.io/blop/

In [1]:
from datetime import datetime
import logging

import bluesky.plan_stubs as bps  # noqa F401
import bluesky.plans as bp  # noqa F401
import databroker  # type: ignore[import-untyped]
import matplotlib.pyplot as plt
from bluesky.callbacks import best_effort
from bluesky.callbacks.tiled_writer import TiledWriter
from bluesky.run_engine import RunEngine
from databroker import Broker
from ophyd.utils import make_dir_tree  # type: ignore[import-untyped]
from tiled.client import from_uri  # type: ignore[import-untyped]
from tiled.client.container import Container
from tiled.server import SimpleTiledServer

from blop.sim import HDF5Handler
from blop.sim.beamline import DatabrokerBeamline, TiledBeamline

# Suppress noisy logs from httpx 
logging.getLogger("httpx").setLevel(logging.WARNING)

DETECTOR_STORAGE = "/mnt/data531/temp/"

[WARNING 11-26 13:09:12] ax.service.utils.with_db_settings_base: Ax currently requires a sqlalchemy version below 2.0. This will be addressed in a future release. Disabling SQL storage in Ax for now, if you would like to use SQL storage please install Ax with mysql extras via `pip install ax-platform[mysql]`.


In [4]:
# removed

# tiled_server = SimpleTiledServer(readable_storage=[DETECTOR_STORAGE])
# tiled_client = from_uri(tiled_server.uri)
# tiled_writer = TiledWriter(tiled_client)

## added
import os
api_key = os.getenv("TILED_SINGLE_USER_API_KEY")
if not api_key:
   raise ValueError("TILED_SINGLE_USER_API_KEY environment variable is not set.")
tiled_client = from_uri("http://192.168.10.155:8000", api_key=api_key)

from bluesky.callbacks.tiled_writer import TiledWriter
tiled_writer = TiledWriter(tiled_client)

def setup_re_env(db_type="default", root_dir="/default/path", method="tiled"):
    RE = RunEngine({})
    bec = best_effort.BestEffortCallback()
    RE.subscribe(bec)
    _ = make_dir_tree(datetime.now().year, base_path=root_dir)
    if method.lower() == "tiled":
        RE.subscribe(tiled_writer)
        return {"RE": RE, "db": tiled_client, "bec": bec}
    elif method.lower() == "databroker":
        db = Broker.named(db_type)
        db.reg.register_handler("HDF5", HDF5Handler, overwrite=True)
        try:
            databroker.assets.utils.install_sentinels(db.reg.config, version=1)
        except Exception:
            pass
        RE.subscribe(db.insert)
        return {"RE": RE, "db": db, "bec": bec, }
    else:
        raise ValueError("The method for data storage used is not supported")


def register_handlers(db, handlers):
    for handler_spec, handler_class in handlers.items():
        db.reg.register_handler(handler_spec, handler_class, overwrite=True)


env = setup_re_env(db_type="temp", root_dir="/tmp/blop/sim", method="tiled")
globals().update(env)
bec.disable_plots()

In [5]:
# if isinstance(db, Container):
#     beamline = TiledBeamline(name="bl")
# elif isinstance(db, databroker.v1.Broker):
#     beamline = DatabrokerBeamline(name="bl")

In [ ]:
import ophyd
from ophyd import EpicsMotor
ophyd.set_cl('caproto')

## added
m101_pitch = EpicsMotor('bl531_esp300:m101_pitch_mm', name='m101_pitch')
#m101_bend  = EpicsMotor('bl531_esp300:m101_bend_um', name='m101_bend')
mono_height = EpicsMotor('bl531_xps1:mono_height_mm', name='mono_height')
mono_angle = EpicsMotor('bl531_xps1:mono_angle_deg', name='mono_angle')

beamstop = ophyd.EpicsSignal('bl201-beamstop:current', name='beamstop')

2025-11-26 13:24:02.338 INFO: connection state changed to connected.
2025-11-26 13:24:02.340 INFO: connection state changed to connected.
2025-11-26 13:24:02.341 INFO: connection state changed to connected.
2025-11-26 13:24:02.342 INFO: connection state changed to connected.
2025-11-26 13:24:02.344 INFO: connection state changed to connected.
2025-11-26 13:24:02.345 INFO: connection state changed to connected.
2025-11-26 13:24:02.347 INFO: connection state changed to connected.
2025-11-26 13:24:02.347 INFO: connection state changed to connected.
2025-11-26 13:24:02.348 INFO: connection state changed to connected.
2025-11-26 13:24:02.349 INFO: connection state changed to connected.
2025-11-26 13:24:02.351 INFO: connection state changed to connected.
2025-11-26 13:24:02.352 INFO: connection state changed to connected.
2025-11-26 13:24:02.353 INFO: connection state changed to connected.
2025-11-26 13:24:02.354 INFO: connection state changed to connected.
2025-11-26 13:24:02.355 INFO: conn

In [9]:
from blop.ax import Agent
from blop.dofs import DOF
from blop.objectives import Objective

dofs = [
    # DOF(movable=beamline.kbv_dsv, type="continuous", search_domain=(-5.0, 5.0)),
    # DOF(movable=beamline.kbv_usv, type="continuous", search_domain=(-5.0, 5.0)),
    # DOF(movable=beamline.kbh_dsh, type="continuous", search_domain=(-5.0, 5.0)),
    # DOF(movable=beamline.kbh_ush, type="continuous", search_domain=(-5.0, 5.0)),
    DOF(movable=mono_angle, type="continuous", search_domain=(20, 22)),
]

objectives = [
    # Objective(name="bl_det_sum", target="max"),
    # Objective(name="bl_det_wid_x", target="min"),
    # Objective(name="bl_det_wid_y", target="min"),
    Objective(name="beamstop", target="min"),
]

agent = Agent(
    #readables=[beamline.det],
    readables=[beamstop],
    dofs=dofs,
    objectives=objectives,
    db=db,
)
agent.configure_experiment(name="test_agent", description="Test the blop 0.8.1")

2025-11-26 13:29:09.623 INFO: Configuring optimization with objective: -beamstop and outcome constraints: []


In [10]:
RE(agent.learn(iterations=2, n=1))

2025-11-26 13:29:34.627 INFO: Executing plan <generator object Agent.learn at 0x71b5e00f7e40>
2025-11-26 13:29:34.629 INFO: Change state on <bluesky.run_engine.RunEngine object at 0x71b5e34c59d0> from 'idle' -> 'running'




Transient Scan ID: 1     Time: 2025-11-26 13:29:34
Persistent Unique Scan ID: '621d6955-078d-469c-a8bd-a6545b75ef71'
New stream: 'primary'
+-----------+------------+------------+------------+
|   seq_num |       time | mono_angle |   beamstop |
+-----------+------------+------------+------------+
|         1 | 13:31:16.9 |    21.0000 |        -10 |
+-----------+------------+------------+------------+
generator list_scan ['621d6955'] (scan num: 1)





Transient Scan ID: 2     Time: 2025-11-26 13:31:18
Persistent Unique Scan ID: '62318ffb-da0c-410d-99c0-3a843485b47f'
New stream: 'primary'
+-----------+------------+------------+------------+
|   seq_num |       time | mono_angle |   beamstop |
+-----------+------------+------------+------------+
|         1 | 13:32:18.1 |    20.4070 |        -10 |
+-----------+------------+------------+------------+
generator list_scan ['62318ffb'] (scan num: 2)





2025-11-26 13:32:19.451 INFO: Change state on <bluesky.run_engine.RunEngine object at 0x71b5e34c59d0> from 'running' -> 'idle'
2025-11-26 13:32:19.452 INFO: Cleaned up from plan <generator object Agent.learn at 0x71b5e00f7e40>


('621d6955-078d-469c-a8bd-a6545b75ef71',
 '62318ffb-da0c-410d-99c0-3a843485b47f')